In [2]:
!pip install -r "requirements.txt"

  Using cached tensorflow-2.22.0rc0-cp314-cp314-win_amd64.whl.metadata (4.6 kB)
Using cached tensorflow-2.22.0rc0-cp314-cp314-win_amd64.whl (384.1 MB)


In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [6]:
# Step 1: Create the dataset
reviews = [
    "Movie was good",
    "Movie was bad",
    "Movie was not that much good"
]

labels = np.array([1, 0, 0])

In [8]:
# Step 2: Convert words into numerical values
tokenizer = Tokenizer()
tokenizer.fit_on_texts(reviews)

sequences = tokenizer.texts_to_sequences(reviews)

print("Word Index:", tokenizer.word_index)
print("Sequences:", sequences)

Word Index: {'movie': 1, 'was': 2, 'good': 3, 'bad': 4, 'not': 5, 'that': 6, 'much': 7}
Sequences: [[1, 2, 3], [1, 2, 4], [1, 2, 5, 6, 7, 3]]


In [9]:
# Step 3: Make all sequences the same length
X = pad_sequences(
    sequences,
    maxlen=6,
    padding="post"
)

y = labels

print("Padded sequences:\n", X)

Padded sequences:
 [[1 2 3 0 0 0]
 [1 2 4 0 0 0]
 [1 2 5 6 7 3]]


In [10]:
# Step 4: Build the RNN model
vocab_size = len(tokenizer.word_index) + 1

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=8,
        mask_zero=True
    ),
    SimpleRNN(8),
    Dense(1, activation="tanh")
])


In [11]:
# Step 5: Compile the model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [12]:
# Step 6: Train the RNN
model.fit(X, y, epochs=200, verbose=0)

print("Training completed!")

C:\Users\Teks\anaconda3\Lib\site-packages\keras\src\layers\layer.py:1078: UserWarning: Layer 'simple_rnn' (of type SimpleRNN) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Training completed!


In [13]:
# Step 7: Predict sentiment for training reviews
predictions = model.predict(X, verbose=0)

for review, probability in zip(reviews, predictions):
    sentiment = "Positive" if probability[0] >= 0.5 else "Negative"

    print("\nReview:", review)
    print("Positive probability:", round(float(probability[0]), 4))
    print("Predicted sentiment:", sentiment)



Review: Movie was good
Positive probability: 0.9155
Predicted sentiment: Positive

Review: Movie was bad
Positive probability: -0.0102
Predicted sentiment: Negative

Review: Movie was not that much good
Positive probability: -0.057
Predicted sentiment: Negative


In [15]:
# Step 8: Predict a new movie review
new_review = ["Movie was Good"]

new_sequence = tokenizer.texts_to_sequences(new_review)
new_padded = pad_sequences(
    new_sequence,
    maxlen=6,
    padding="post"
)

probability = model.predict(new_padded, verbose=0)[0][0]

print("\nNew review:", new_review[0])
print("Positive probability:", round(float(probability), 4))
print(
    "Sentiment:",
    "Positive" if probability >= 0.5 else "Negative"
)



New review: Movie was Good
Positive probability: 0.9155
Sentiment: Positive
